# Experiment: Full Pipeline End-to-End

Objective:
- Run the full operational flow from data preparation through metrics/figures.
- Support both dry-run mode and executable mode when deferred functions are implemented.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


def read_rows_csv(path: Path) -> list[dict[str, str]]:
    with path.open('r', newline='', encoding='utf-8') as f:
        return [dict(r) for r in csv.DictReader(f)]


def count_rows(path: Path) -> int:
    return len(read_rows_csv(path))


## Parameters

Set `EXECUTE_DETECTORS=True` and `INCLUDE_SRM=True` to trigger SRM auto-training (when SRM deferred functions are implemented).


In [ ]:
from src.data.download_real_covers import download_real_covers
from src.data.generate_ml_covers import generate_ml_covers_from_prompts
from src.data.merge_covers_master import merge_covers_master
from src.pipeline.config import PipelineConfig
from src.pipeline.runner import PipelineRunner

RUN_REAL_DOWNLOAD = False
RUN_ML_GENERATION = False
ML_ENGINE = 'stub'  # use 'diffusers' for real generation

EXECUTE_EMBEDDINGS = False
EXECUTE_DETECTORS = False
INCLUDE_SRM = False  # set True for SRM auto-train path during detector execution
SKIP_UNIMPLEMENTED = True
GENERATE_FIGURES = True

REAL_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_real.csv'
ML_A_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_ml_a.csv'
ML_B_MANIFEST = PROJECT_ROOT / 'data/manifests/covers_master_ml_b.csv'
COVERS_MASTER = PROJECT_ROOT / 'data/manifests/covers_master.csv'
PROMPTS_CSV = PROJECT_ROOT / 'data/manifests/generation_prompts.csv'


## Optional Data Preparation (Real + ML + Merge)

In [ ]:
if RUN_REAL_DOWNLOAD:
    out_real = download_real_covers(project_root=PROJECT_ROOT)
    print('Real download outputs:')
    for k, v in out_real.items():
        print(f'  {k}: {v}')

if RUN_ML_GENERATION:
    out_ml = generate_ml_covers_from_prompts(
        project_root=PROJECT_ROOT,
        prompts_csv=PROMPTS_CSV,
        engine=ML_ENGINE,
        max_groups=None,
    )
    print('ML generation outputs:')
    for k, v in out_ml.items():
        print(f'  {k}: {v}')

if REAL_MANIFEST.exists() and ML_A_MANIFEST.exists() and ML_B_MANIFEST.exists():
    merged = merge_covers_master(
        project_root=PROJECT_ROOT,
        real_manifest=REAL_MANIFEST,
        ml_a_manifest=ML_A_MANIFEST,
        ml_b_manifest=ML_B_MANIFEST,
        output_manifest=COVERS_MASTER,
        expected_groups=500,
    )
    print(f'Merged covers manifest: {merged}')
else:
    print('Skipping merge; required source manifests are missing.')


## Run Full Pipeline via Runner API

SRM note: when detectors execute with SRM enabled and no external SRM provider, the runner auto-trains SRM per fold/method before scoring test rows.


In [ ]:
if not COVERS_MASTER.exists():
    raise FileNotFoundError(f'Missing {COVERS_MASTER}. Create/merge covers first.')

cfg = PipelineConfig.from_project_root(PROJECT_ROOT)
runner = PipelineRunner(cfg)

outputs = runner.run_full_pipeline(
    covers_manifest_path=COVERS_MASTER,
    execute_embeddings=EXECUTE_EMBEDDINGS,
    execute_detectors=EXECUTE_DETECTORS,
    include_srm=INCLUDE_SRM,
    skip_unimplemented=SKIP_UNIMPLEMENTED,
    generate_figures=GENERATE_FIGURES,
)

print('Pipeline outputs:')
for k, v in outputs.items():
    print(f'  {k}: {v}')


In [ ]:
print('Key row counts:')
print(f'  covers_master: {count_rows(PROJECT_ROOT / "data/manifests/covers_master.csv")}')
print(f'  payload_manifest: {count_rows(PROJECT_ROOT / "data/manifests/payload_manifest.csv")}')
print(f'  stego_manifest: {count_rows(PROJECT_ROOT / "data/manifests/stego_manifest.csv")}')
print(f'  predictions: {count_rows(PROJECT_ROOT / "results/predictions/predictions.csv")}')

metrics_files = [
    PROJECT_ROOT / 'results/metrics/fold_metrics.csv',
    PROJECT_ROOT / 'results/metrics/condition_metrics.csv',
    PROJECT_ROOT / 'results/metrics/source_contrasts.csv',
    PROJECT_ROOT / 'results/metrics/pooled_summary.csv',
]
for path in metrics_files:
    print(f'  {path.relative_to(PROJECT_ROOT)} exists={path.exists()} rows={count_rows(path) if path.exists() else 0}')


In [ ]:
from IPython.display import Image, display

fig_paths = [
    PROJECT_ROOT / 'results/figures/auc_by_source_detector.png',
    PROJECT_ROOT / 'results/figures/auc_by_method_detector.png',
]
for path in fig_paths:
    if path.exists():
        print(f'Displaying {path.relative_to(PROJECT_ROOT)}')
        display(Image(filename=str(path)))
    else:
        print(f'Figure not found: {path.relative_to(PROJECT_ROOT)}')


## Equivalent One-Command CLI

```bash
python3 -m src.pipeline.cli --project-root . run-all --covers-manifest data/manifests/covers_master.csv --generate-figures
```

For full execution once deferred methods are implemented:

```bash
python3 -m src.pipeline.cli --project-root . run-all --covers-manifest data/manifests/covers_master.csv --execute-embeddings --execute-detectors --generate-figures
```